# ฮองเรียน — เซิร์ฟเวอร์เสียงพากย์ระดับสตูดิโอ & ASR บน Colab GPU (A100)

โน้ตบุ๊กนี้ย้าย **โมเดลเสียงพากย์ไทยตัวท็อปและระบบถอดเสียง (ASR)** มารันบน **Colab A100** ส่วนการแปลซับยังใช้ **`agy`** ในเครื่องของคุณเหมือนเดิม:
- **ThonburianTTS (`megaF5`)** — โมเดล F5-TTS ภาษาไทยโดย BioDatLab (มจธ.) ออกเสียงศัพท์เทคนิคอังกฤษปนไทยชัดมาก + โคลนเสียงผู้สอนได้
- **OpenJaiTTS (`v1.0`) / F5-TTS-THAI** — โมเดล F5-TTS ฝึกจากเสียงไทยระดับห้องอัด (หากยังไม่ใส่ `HF_TOKEN` จะสลับใช้ `VIZINTZOR/F5-TTS-THAI` ให้อัตโนมัติ)
- **OmniVoice Thai** — โมเดล Diffusion-LM เลือกโทนเสียงชาย/หญิง หรือโคลนเสียงผู้สอน
- **Meta SeamlessM4T-v2-Large (2.3B)** — รุ่นใหญ่สุดสำหรับโหมด **เริ่มทันที** (แปลและพากย์สดจากต้นฉบับทันที)
- **Thai Whisper Large-v3 Combined & Whisper Large-v3 Turbo** — ถอดความไฟล์เสียงตัวอย่างโคลนเสียง และสร้างซับจากวิดีโอที่ไม่มีซับ (`/v1/asr`)

**วิธีใช้**
1. เลือก `Runtime → Change runtime type → A100 GPU` แล้วกด `Runtime → Run all`
2. รอจนเซลล์ที่ 4 พิมพ์ค่า `UDTH_A100_URL` และ `UDTH_A100_KEY` ออกมา
3. คัดลอกไปวางท้ายไฟล์ `.env` ในเครื่องคุณ แล้วปิด-เปิด `start-server.cmd` ใหม่
4. ถ้า Colab หลุดหรือปิด เซิร์ฟเวอร์ในเครื่องจะสลับไปใช้เสียงสำรอง (Local GPU / Microsoft Edge Neural) เองโดยอัตโนมัติ วิดีโอไม่ค้าง

## 1. ตั้งค่าโมเดลที่จะโหลดขึ้นการ์ดจอ A100
A100 (40GB / 80GB) มี VRAM เหลือเฟือสำหรับโหลดทุกโมเดลพร้อมกัน (ใช้รวมกันราว 12–16 GB เท่านั้น)

In [1]:
ENABLE_THONBURIAN = True   # ThonburianTTS megaF5 (แนะนำสำหรับคอร์สเขียนโปรแกรม)
ENABLE_OPENJAI    = True   # OpenJaiTTS v1.0 / F5-TTS-THAI สตูดิโอ
ENABLE_OMNIVOICE  = True   # OmniVoice Thai 0.6B
ENABLE_SEAMLESS   = True   # Meta SeamlessM4T-v2-large (2.3B)
ENABLE_ASR        = True   # Whisper Thai Large-v3 Combined + Whisper Large-v3 Turbo

SEAMLESS_MODEL = "facebook/seamless-m4t-v2-large"
PORT = 8001

## 2. ติดตั้งไลบรารีเสียงแบบด่วน (~25 วินาที)
*(หมายเหตุ: ติดตั้ง `f5-tts` ด้วย `--no-deps` และจำลอง `cached_path` ผ่าน `huggingface_hub` โดยตรง เพื่อป้องกัน `pip` ติดลูป Backtracking หรือเรียกหา `boto3`)*

In [2]:
!apt-get -qq update && apt-get -qq install -y ffmpeg
!pip install -q omnivoice vocos pythainlp pydub einops ema-pytorch torchdiffeq x-transformers rjieba pypinyin jieba
!pip install -q --no-deps f5-tts
!rm -rf /content/thonburian-tts && git clone -q https://github.com/biodatlab/thonburian-tts.git /content/thonburian-tts
!wget -q -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x /usr/local/bin/cloudflared
!nvidia-smi --query-gpu=name,memory.total --format=csv

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.6/107.6 kB 12.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 124.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.5/168.5 kB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.9/21.9 MB 108.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 122.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 840.2/840.2 kB 55.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.8/163.8 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.5/

## 3. โหลดโมเดลขึ้น A100 และเปิดเซิร์ฟเวอร์ `/v1/tts` + `/v1/asr`

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=SyntaxWarning)

import base64, hashlib, importlib.util, io, json, os, secrets, subprocess, sys, tempfile, threading, time, traceback, types
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from pathlib import Path

# Auto-bootstrap f5-tts & thonburian-tts (using --no-deps to avoid pip resolver backtracking)
if importlib.util.find_spec("f5_tts") is None or importlib.util.find_spec("pythainlp") is None:
    print("กำลังติดตั้งไลบรารีเสียงแบบด่วน (--no-deps สำหรับ f5-tts) ...")
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "vocos", "pythainlp", "pydub", "einops", "ema-pytorch",
        "torchdiffeq", "x-transformers", "rjieba", "pypinyin", "jieba",
    ])
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", "--no-deps", "f5-tts",
    ])

# Lightweight shims for unused top-level imports in flowtts & f5_tts (avoids boto3, wandb, hydra, datasets)
def _hf_cached_path(url_or_path, *args, **kwargs):
    s = str(url_or_path)
    if s.startswith("hf://"):
        from huggingface_hub import hf_hub_download
        parts = s[5:].split("/")
        repo_id = f"{parts[0]}/{parts[1]}"
        filename = "/".join(parts[2:])
        return Path(hf_hub_download(repo_id=repo_id, filename=filename, cache_dir=kwargs.get("cache_dir")))
    return Path(s)

_cp_mod = types.ModuleType("cached_path")
_cp_mod.cached_path = _hf_cached_path
sys.modules["cached_path"] = _cp_mod

if importlib.util.find_spec("wandb") is None:
    sys.modules["wandb"] = types.ModuleType("wandb")
if importlib.util.find_spec("hydra") is None:
    _hydra = types.ModuleType("hydra")
    _hydra_utils = types.ModuleType("hydra.utils")
    _hydra_utils.get_class = lambda *a, **k: None
    _hydra.utils = _hydra_utils
    sys.modules["hydra"] = _hydra
    sys.modules["hydra.utils"] = _hydra_utils
if importlib.util.find_spec("omegaconf") is None:
    _oc = types.ModuleType("omegaconf")
    _oc.OmegaConf = object
    sys.modules["omegaconf"] = _oc
if importlib.util.find_spec("datasets") is None:
    _ds = types.ModuleType("datasets")
    _ds.Dataset = object
    _ds.load_from_disk = lambda *a, **k: None
    sys.modules["datasets"] = _ds

if not Path("/content/thonburian-tts/flowtts").is_dir():
    print("กำลังดึงซอร์สโค้ด flowtts จาก biodatlab/thonburian-tts ...")
    subprocess.check_call(["git", "clone", "-q", "https://github.com/biodatlab/thonburian-tts.git", "/content/thonburian-tts"])
if "/content/thonburian-tts" not in sys.path:
    sys.path.insert(0, "/content/thonburian-tts")

import numpy as np, soundfile as sf, torch

try:
    from google.colab import userdata
    hf_tok = userdata.get("HF_TOKEN")
    if hf_tok:
        os.environ["HF_TOKEN"] = hf_tok
except Exception:
    pass

API_KEY = globals().get("API_KEY") or secrets.token_urlsafe(32)
WORK_DIR = Path("/content/hongrian_a100")
REF_DIR = WORK_DIR / "refs"
REF_DIR.mkdir(parents=True, exist_ok=True)
DEFAULT_REF_TEXT = "สวัสดีครับ วันนี้เราจะมาเรียนรู้การสร้างเว็บไซต์ด้วยภาษา HTML และ CSS กันนะครับ"
DEFAULT_REF_WAV = REF_DIR / "default_ref.wav"

LOADED = globals().get("LOADED") or {}
GPU_LOCK = threading.Lock()

# 1. OmniVoice Thai + สร้างไฟล์เสียงอ้างอิงมาตรฐาน
if "omni" not in globals():
    omni = None
if ENABLE_OMNIVOICE and omni is None:
    from omnivoice import OmniVoice
    print("กำลังโหลด OmniVoice Thai ...")
    omni = OmniVoice.from_pretrained("hotdogs/omnivoice-thai").to("cuda:0")
    if getattr(omni, "audio_tokenizer", None) is not None:
        omni.audio_tokenizer = omni.audio_tokenizer.to("cuda:0")
    ref_samples = np.asarray(omni.generate(text=DEFAULT_REF_TEXT, instruct="male, young adult, moderate pitch")[0], dtype=np.float32).reshape(-1)
    sf.write(str(DEFAULT_REF_WAV), ref_samples, 24000, format="WAV")
    LOADED["omnivoice"] = "hotdogs/omnivoice-thai"

# 2. ThonburianTTS megaF5 & OpenJaiTTS v1.0 (with keyword args for FlowTTSPipeline)
if "flow_pipes" not in globals():
    flow_pipes = {}
if ENABLE_THONBURIAN or ENABLE_OPENJAI:
    from flowtts.inference import FlowTTSPipeline, ModelConfig, AudioConfig
    def _load_flow(name, ckpt, vocab, fallback=None):
        if name in flow_pipes:
            return
        try:
            print(f"กำลังโหลด {name} ({ckpt}) ...")
            local_ckpt = str(_hf_cached_path(ckpt))
            local_vocab = str(_hf_cached_path(vocab))
            flow_pipes[name] = FlowTTSPipeline(
                model_config=ModelConfig(
                    language="th",
                    model_type="F5",
                    checkpoint=local_ckpt,
                    vocab_file=local_vocab,
                    vocoder="vocos",
                    device="cuda",
                ),
                audio_config=AudioConfig(silence_threshold=-45, cfg_strength=2.5, speed=1.0),
                temp_dir=str(WORK_DIR / f"temp_{name}"),
            )
            LOADED[name] = ckpt
        except Exception as exc:
            if fallback is not None:
                fb_ckpt, fb_vocab = fallback
                print(f"  หมายเหตุ: โหลด {ckpt} ไม่ได้ ({exc}) — สลับใช้โมเดลเสียงไทยสตูดิโอ {fb_ckpt} แทน")
                _load_flow(name, fb_ckpt, fb_vocab, fallback=None)
            else:
                print(f"ข้าม {name}: {exc}")

    if ENABLE_THONBURIAN:
        _load_flow(
            "thonburian",
            "hf://biodatlab/ThonburianTTS/megaF5/mega_f5_last.safetensors",
            "hf://biodatlab/ThonburianTTS/megaF5/mega_vocab.txt",
        )
    if ENABLE_OPENJAI:
        _load_flow(
            "openjai",
            "hf://JTS-AI/OpenJaiTTS-v1.0/model.pt",
            "hf://JTS-AI/OpenJaiTTS-v1.0/vocab.txt",
            fallback=("hf://VIZINTZOR/F5-TTS-THAI/model_600000_FP16.pt", "hf://VIZINTZOR/F5-TTS-THAI/vocab.txt"),
        )

# 3. Meta SeamlessM4T-v2-Large
if "seamless_model" not in globals():
    seamless_proc, seamless_model = None, None
if ENABLE_SEAMLESS and seamless_model is None:
    import transformers
    from transformers import AutoProcessor
    print(f"กำลังโหลด {SEAMLESS_MODEL} ...")
    seamless_proc = AutoProcessor.from_pretrained(SEAMLESS_MODEL)
    cls = transformers.SeamlessM4Tv2Model if "v2" in SEAMLESS_MODEL.lower() else transformers.SeamlessM4TModel
    seamless_model = cls.from_pretrained(SEAMLESS_MODEL, torch_dtype=torch.float16).to("cuda:0")
    LOADED["seamless"] = SEAMLESS_MODEL

# 4. Whisper ASR (Thai Whisper Large-v3 Combined + Whisper Large-v3 Turbo)
if "asr_pipes" not in globals():
    asr_pipes = {}
if ENABLE_ASR:
    from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline
    for key, mid in [("th", "biodatlab/whisper-th-large-v3-combined"), ("multi", "openai/whisper-large-v3-turbo")]:
        if key in asr_pipes:
            continue
        try:
            print(f"กำลังโหลด ASR ({mid}) ...")
            m = AutoModelForSpeechSeq2Seq.from_pretrained(mid, torch_dtype=torch.float16, low_cpu_mem_usage=True).to("cuda:0")
            p = AutoProcessor.from_pretrained(mid)
            asr_pipes[key] = pipeline("automatic-speech-recognition", model=m, tokenizer=p.tokenizer,
                                      feature_extractor=p.feature_extractor, chunk_length_s=30, torch_dtype=torch.float16, device="cuda:0")
            LOADED[f"asr_{key}"] = mid
        except Exception as exc:
            print(f"ข้าม ASR {mid}: {exc}")

def _resolve_ref(body):
    ref_id = body.get("ref_id")
    if not ref_id:
        return str(DEFAULT_REF_WAV), DEFAULT_REF_TEXT, None
    wav_path = REF_DIR / f"{ref_id}.wav"
    txt_path = REF_DIR / f"{ref_id}.txt"
    if body.get("ref_audio_b64"):
        wav_path.write_bytes(base64.b64decode(body["ref_audio_b64"]))
    if not wav_path.is_file():
        raise KeyError("missing_ref")
    ref_text = (body.get("ref_text") or "").strip()
    transcribed_now = None
    if not ref_text and txt_path.is_file():
        ref_text = txt_path.read_text(encoding="utf-8").strip()
    if not ref_text:
        pipe = asr_pipes.get("th") or asr_pipes.get("multi")
        if pipe is not None:
            ref_text = pipe(str(wav_path), generate_kwargs={"language": "th"})["text"].strip()
        else:
            ref_text = DEFAULT_REF_TEXT
        txt_path.write_text(ref_text, encoding="utf-8")
        transcribed_now = ref_text
    return str(wav_path), ref_text, transcribed_now

class A100Handler(BaseHTTPRequestHandler):
    def log_message(self, *args): pass
    def _send(self, code, payload):
        data = json.dumps(payload, ensure_ascii=False).encode("utf-8")
        self.send_response(code)
        self.send_header("Content-Type", "application/json; charset=utf-8")
        self.send_header("Content-Length", str(len(data)))
        self.end_headers()
        self.wfile.write(data)
    def _auth(self):
        if self.headers.get("Authorization") != f"Bearer {API_KEY}":
            self._send(401, {"error": "invalid API key"})
            return False
        return True
    def do_GET(self):
        if not self._auth(): return
        if self.path == "/v1/health":
            self._send(200, {"ok": True, "gpu": torch.cuda.get_device_name(0), "models": LOADED})
            return
        self._send(404, {"error": "not found"})
    def do_POST(self):
        if not self._auth(): return
        body = json.loads(self.rfile.read(int(self.headers.get("Content-Length", "0"))).decode("utf-8"))
        try:
            if self.path == "/v1/tts":
                engine = body.get("engine", "thonburian")
                text = body.get("text", "").strip()
                with GPU_LOCK:
                    if engine in ("thonburian", "openjai"):
                        pipe = flow_pipes.get(engine) or flow_pipes.get("thonburian")
                        if pipe is None: raise RuntimeError(f"ยังไม่ได้โหลดโมเดล {engine}")
                        ref_wav, ref_txt, new_txt = _resolve_ref(body)
                        if hasattr(pipe, "generate"):
                            audio, sr = pipe.generate(reference_audio=ref_wav, reference_text=ref_txt, gen_text=text)
                            buf = io.BytesIO()
                            sf.write(buf, np.asarray(audio, dtype=np.float32).reshape(-1), sr, format="WAV")
                            wav_bytes = buf.getvalue()
                        else:
                            tmp = WORK_DIR / "_tmp.wav"
                            pipe(text=text, ref_voice=ref_wav, ref_text=ref_txt, output_file=str(tmp))
                            wav_bytes = tmp.read_bytes()
                        self._send(200, {"audio_b64": base64.b64encode(wav_bytes).decode("ascii"), "ref_text": new_txt})
                        return
                    if engine == "omnivoice":
                        if omni is None: raise RuntimeError("ยังไม่ได้โหลด OmniVoice")
                        if body.get("ref_id"):
                            ref_wav, ref_txt, new_txt = _resolve_ref(body)
                            prompt = omni.create_voice_clone_prompt(ref_audio=ref_wav, ref_text=ref_txt)
                            audio = omni.generate(text=text, voice_clone_prompt=prompt)[0]
                        else:
                            new_txt = None
                            audio = omni.generate(text=text, instruct=body.get("instruct") or "male, low pitch")[0]
                        buf = io.BytesIO()
                        sf.write(buf, np.asarray(audio, dtype=np.float32).reshape(-1), 24000, format="WAV")
                        self._send(200, {"audio_b64": base64.b64encode(buf.getvalue()).decode("ascii"), "ref_text": new_txt})
                        return
                    if engine == "seamless":
                        if seamless_model is None: raise RuntimeError("ยังไม่ได้โหลด SeamlessM4T")
                        src = body.get("source_lang") or "tha"
                        inputs = seamless_proc(text=text, src_lang=src, return_tensors="pt").to("cuda:0")
                        with torch.no_grad():
                            out = seamless_model.generate(**inputs, tgt_lang="tha", generate_speech=True, return_intermediate_token_ids=True)
                        audio_np = out.waveform[0].cpu().squeeze().to(torch.float32).numpy()
                        spoken = ""
                        if src != "tha" and out.sequences is not None:
                            spoken = seamless_proc.decode(out.sequences[0].tolist(), skip_special_tokens=True).replace("ํา", "ำ").strip()
                        buf = io.BytesIO()
                        sf.write(buf, audio_np, 16000, format="WAV")
                        self._send(200, {"audio_b64": base64.b64encode(buf.getvalue()).decode("ascii"), "text": spoken})
                        return
            if self.path == "/v1/asr":
                lang = body.get("lang")
                pipe = asr_pipes.get("th" if lang == "th" else "multi") or next(iter(asr_pipes.values()), None)
                if pipe is None: raise RuntimeError("ยังไม่ได้โหลดโมเดล ASR")
                tmp = WORK_DIR / "_asr_in.wav"
                tmp.write_bytes(base64.b64decode(body["audio_b64"]))
                with GPU_LOCK:
                    kw = {"language": lang} if lang else {}
                    res = pipe(str(tmp), return_timestamps=bool(body.get("timestamps", True)), generate_kwargs=kw)
                cues = []
                for c in res.get("chunks") or []:
                    ts = c.get("timestamp") or (0.0, 0.0)
                    s = float(ts[0] or 0.0)
                    e = float(ts[1] if ts[1] is not None else s + 3.0)
                    cues.append({"start": round(s, 2), "end": round(e, 2), "text": (c.get("text") or "").strip()})
                self._send(200, {"text": (res.get("text") or "").strip(), "lang": lang or "en", "cues": cues})
                return
            self._send(404, {"error": "not found"})
        except KeyError as exc:
            if str(exc).strip("'") == "missing_ref":
                self._send(428, {"error": "missing_ref"})
            else:
                self._send(500, {"error": str(exc)})
        except Exception as exc:
            traceback.print_exc()
            self._send(500, {"error": str(exc)})

try:
    http_server.shutdown()
    http_server.server_close()
except Exception:
    pass
http_server = ThreadingHTTPServer(("127.0.0.1", PORT), A100Handler)
threading.Thread(target=http_server.serve_forever, daemon=True).start()
print("เซิร์ฟเวอร์เสียง A100 พร้อมทำงาน:", LOADED)

กำลังโหลด OmniVoice Thai ...


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 14 files:   0%|          | 0/14 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/313 [00:00<?, ?it/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

...

Loading weights:   0%|          | 0/2232 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/9.91M [00:00<?, ?B/s]

กำลังโหลด ASR (biodatlab/whisper-th-large-v3-combined) ...


config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


## 4. เปิด Cloudflare Tunnel แล้วคัดลอกค่าไปวางใน `.env`

In [ ]:
import re, subprocess

tunnel = subprocess.Popen(["cloudflared", "tunnel", "--url", f"http://127.0.0.1:{PORT}", "--no-autoupdate"],
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
public = None
for line in tunnel.stdout:
    found = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if found:
        public = found.group(0)
        break

print("วางบรรทัดด้านล่างนี้ท้ายไฟล์ .env ในเครื่องคุณ แล้วปิด-เปิด start-server.cmd ใหม่:\n")
print(f"UDTH_A100_URL={public}")
print(f"UDTH_A100_KEY={API_KEY}")
print("UDTH_PREFETCH_VOICE=thonburian:default")

## 5. ทดสอบสังเคราะห์เสียงพากย์ผ่าน Tunnel

In [ ]:
import urllib.request
from IPython.display import Audio, display

time.sleep(5)
req = urllib.request.Request(
    f"{public}/v1/tts",
    data=json.dumps({"engine": "thonburian", "voice": "thonburian:default",
                     "text": "สวัสดีครับ วันนี้เราจะเชื่อมต่อฐานข้อมูล PostgreSQL ผ่าน Node.js กันครับ"}).encode("utf-8"),
    headers={"Content-Type": "application/json", "Authorization": f"Bearer {API_KEY}"},
)
t0 = time.time()
res = json.load(urllib.request.urlopen(req, timeout=60))
wav = base64.b64decode(res["audio_b64"])
print(f"สร้างเสียงผ่าน Tunnel สำเร็จใน {time.time() - t0:.2f} วินาที ({len(wav):,} bytes)")
display(Audio(wav))

## 6. รันค้างไว้ระหว่างเรียน (กด Stop เมื่อเลิกใช้งาน)

In [ ]:
while True:
    alive = tunnel.poll() is None
    print(time.strftime("%H:%M:%S"), "A100 Voice Server พร้อมใช้งาน" if alive else "Tunnel หยุดทำงาน", end="\r")
    if not alive:
        break
    time.sleep(60)

In [ ]:
import time
s = "ตอนนี้เราจะมาดูว่า CSS cascade ทำงานอย่างไรกับสไตล์ที่ซ้อนกันหลายชั้น"
for i in range(3): t = time.time(); a = omni.generate(text=s, instruct="male, low pitch")[0]; print("omni gen", round(time.time() - t, 2), "s for", round(len(a) / 24000, 1), "s audio")
print(omni.device, next(omni.parameters()).dtype, torch.cuda.get_device_name(0))

In [ ]:
tunnel = subprocess.Popen(["cloudflared", "tunnel", "--url", f"http://127.0.0.1:{PORT}", "--no-autoupdate"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, start_new_session=True)
public = next(m.group(0) for line in tunnel.stdout for m in [re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)] if m)
print("UDTH_A100_URL=" + public)

In [ ]:
def _to_ogg(b64): x, sr = sf.read(io.BytesIO(base64.b64decode(b64))); out = io.BytesIO(); sf.write(out, x, sr, format="OGG", subtype="VORBIS"); return {"audio_b64": base64.b64encode(out.getvalue()).decode("ascii"), "format": "ogg"}
_orig_send = globals().get("_orig_send") or A100Handler._send
A100Handler._send = lambda self, code, payload: _orig_send(self, code, {**payload, **_to_ogg(payload["audio_b64"])} if self.headers.get("X-Audio-Format") == "ogg" and isinstance(payload, dict) and payload.get("audio_b64") else payload)
_orig_prompt = globals().get("_orig_prompt") or omni.create_voice_clone_prompt; _prompts = {}
omni.create_voice_clone_prompt = lambda ref_audio, ref_text=None, **kw: _prompts.setdefault((ref_audio, ref_text, os.path.getmtime(ref_audio)), None) or _prompts.__setitem__((ref_audio, ref_text, os.path.getmtime(ref_audio)), _orig_prompt(ref_audio=ref_audio, ref_text=ref_text, **kw)) or _prompts[(ref_audio, ref_text, os.path.getmtime(ref_audio))]
print("patched; ogg ok:", "OGG" in sf.available_formats())